# Predict unseen perturbations in Tian CRISPRi

Refit the Figure 2 single-gene model on S3, split 101. The example uses the original training response means, control means, output-gene mask and 12 training-normalized prior kernels. Each query is a held-out perturbation in one cellular context. A CPU is sufficient.

In [1]:
from pathlib import Path
import sys
import numpy as np

root = Path.cwd().resolve()
if not (root / 'kipert').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from tutorial.run import DATA, refit, verify

### Inspect the held-out queries

The gene axis is shared across the four contexts. A target can occur in more than one context.

In [2]:
with np.load(DATA / 'axes.npz') as axes:
    targets, genes, contexts = axes['target'], axes['genes'], axes['context']
print(f'{len(targets)} query rows, {len(set(targets))} distinct targets, {len(set(contexts))} contexts, {len(genes):,} genes')

33 query rows, 19 distinct targets, 4 contexts, 14,581 genes


### Fit the model and predict

Within each context, KiPERT estimates alignment weights from the training responses. It combines the resulting kernel with the training response residuals using the ridge penalty chosen on validation data. The returned matrix follows the query and gene order above.

In [3]:
prediction, weights = refit()
print('Prediction matrix:', prediction.shape)
print(weights.pivot(index='prior', columns='context', values='weight').round(4).to_string())

Prediction matrix: (33, 14581)
context            1       2       3       4
prior                                       
CORUM         0.0962  0.0921  0.0718  0.0828
CollecTRI     0.0664  0.0803  0.0810  0.0848
DepMap        0.0490  0.0430  0.0549  0.0417
GO_BP         0.1336  0.1251  0.1033  0.1402
GO_CC         0.1025  0.0832  0.0936  0.0957
GO_MF         0.1051  0.1279  0.1018  0.0853
GenePT        0.0725  0.0852  0.0864  0.0805
HPA           0.0545  0.0506  0.0570  0.0440
HPO           0.0849  0.0880  0.0876  0.1041
Reactome      0.0951  0.0874  0.0986  0.0814
STRING        0.0643  0.0608  0.0796  0.0783
WikiPathways  0.0759  0.0763  0.0845  0.0811


### Inspect one predicted response

Subtract the matched control mean to express the first query as an expression change. The five genes below have the largest absolute predicted changes for this query.

In [4]:
with np.load(DATA / 'axes.npz') as axes:
    context_id = axes['contexts'].tolist().index(contexts[0])
with np.load(DATA / f'context_{context_id}.npz') as inputs:
    delta = prediction[0] - inputs['control']
print(f'Target {targets[0]}, context {contexts[0]}')
for j in np.argsort(np.abs(delta))[-5:][::-1]:
    print(f'{genes[j]}  {delta[j]:+.4f}')

Target ENSG00000005339, context 1
ENSG00000160753  +0.1777
ENSG00000197451  -0.1765
ENSG00000104979  +0.1759
ENSG00000106682  -0.1750
ENSG00000122484  -0.1721


### Verify against the original run

Load the archived prediction only after fitting and check every entry and both query identifiers. This checks numerical replay of one split. Benchmark performance uses held-out experimental observations and is summarized separately in the result ledger.

In [5]:
error = verify(prediction)
print(f'Maximum absolute difference from the archived prediction: {error:.3g}')

Maximum absolute difference from the archived prediction: 0
